In [49]:
import pandas as pd
import requests
import time
import datetime
import os
import csv

try:
    from google.colab import userdata
    my_api_key = userdata.get('RapidAPI')
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    my_api_key = os.getenv("RAPIDAPI_KEY")

def scraping(query_list, output_excel, target_count=5, is_official_account=False, start_date=None, end_date=None):

    seen_video_ids = set()
    all_data_rows = []  

  
    start_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d') if start_date else None
    end_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d') if end_date else None

    if is_official_account:
        print(f"\nSearch by account : {len(query_list)} accounts")
        url = "https://tiktok-api6.p.rapidapi.com/user/videos"
        headers = {
            "x-rapidapi-host": "tiktok-api6.p.rapidapi.com",
            "x-rapidapi-key": my_api_key
        }
        excel_columns = ["target_account", "video_id", "real_video_url", "create_time", 
                       "publish_date", "video_text", "author_name", "play_count", "hearts_count"]
    else:
        print(f"\nSearch by keyword: {len(query_list)} keywords")
        url = "https://tiktok-scraper7.p.rapidapi.com/feed/search"
        headers = {
            "x-rapidapi-host": "tiktok-scraper7.p.rapidapi.com",
            "x-rapidapi-key": my_api_key
        }
        excel_columns = ["search_keyword", "video_id", "real_video_url", "create_time", 
                       "publish_date", "video_text", "author_name", "play_count", "digg_count"]

    for item in query_list:
        print(f"Looking for : {item}")
        total_scraped = 0
        current_cursor = "0"
        has_more = True
        retry_count = 0
        stop_this_account = False  

        while has_more and total_scraped < target_count and not stop_this_account:

            if is_official_account:
        
                querystring = {"username": item, "cursor": current_cursor}
            else:
                querystring = {"keywords": item, "region": "be", "count": "30", "cursor": current_cursor, "publish_time": "0", "sort_type": "0"}

            try:
                response = requests.get(url, headers=headers, params=querystring)
                
                if not is_official_account and response.status_code == 429:
                    retry_count += 1
                    if retry_count > 3: break
                    time.sleep(15)
                    continue
                elif response.status_code != 200:
                    print(f"API Error: {response.status_code}")
                    break

                data = response.json()
                
                if is_official_account:
                    videos_list = data.get("videos", []) 
                else:
                    videos_list = data.get("data", {}).get("videos", []) or data.get("data", [])
                
                if not videos_list: break

                for video in videos_list:
                    if not isinstance(video, dict): continue

                    raw_id = video.get("video_id") or video.get("aweme_id") or video.get("id")
                    video_id = str(raw_id) if raw_id else ""

                    if video_id and video_id not in seen_video_ids:
                        seen_video_ids.add(video_id)
                        
                        raw_time = video.get("create_time")
                        if not raw_time: continue
                        
                       
                        video_dt = datetime.datetime.fromtimestamp(int(raw_time))
                        formatted_date = video_dt.strftime('%Y-%m-%d %H:%M:%S')
                        stats = video.get("statistics", {})

                        if end_dt and video_dt > end_dt:
                   
                            continue
                        
                        if start_dt and video_dt < start_dt:
                           
                            if is_official_account:
                                stop_this_account = True
                                break
                            else:
                                continue  

                        text = str(video.get("title") or video.get("desc") or video.get("description") or "").strip()
                        
                        if is_official_account:
                            real_url = f"https://www.tiktok.com/@{item}/video/{video_id}"
                            all_data_rows.append([
                                item, video_id, real_url, raw_time, formatted_date, text, 
                                item, stats.get("number_of_plays", 0) or stats.get("play_count", 0), stats.get("number_of_hearts", 0) or stats.get("digg_count", 0)
                            ])
                        else:
                            actual_author = video.get("author", {}).get("unique_id") or video.get("author", {}).get("nickname") or "user"
                            real_url = f"https://www.tiktok.com/@{actual_author}/video/{video_id}"
                            all_data_rows.append([
                                item, video_id, real_url, raw_time, formatted_date, text, 
                                actual_author, stats.get("play_count", 0), stats.get("digg_count", 0)
                            ])
                                
                        total_scraped += 1
                        if total_scraped >= target_count: break

                if stop_this_account: break

              
                if is_official_account:
                    next_cursor = data.get("continuation_token")
                    if next_cursor and str(next_cursor) != "0" and str(next_cursor) != current_cursor:
                        current_cursor = str(next_cursor)
                    else:
                        has_more = False
                else:
                    meta_data = data.get("data", {})
                    current_cursor = str(meta_data.get("cursor", int(current_cursor) + 30))
                    has_more = meta_data.get("hasMore", False) or meta_data.get("has_more", False)

            except Exception as e:
                print(f"Error: {e}")
                break
            
            time.sleep(3)

    if all_data_rows:
        df_output = pd.DataFrame(all_data_rows, columns=excel_columns)
        df_output.to_excel(output_excel, index=False)
        print(f" {len(df_output)} videos, saved as：{output_excel}")
    else:
        print(f"no videos")

In [41]:
def main():
  
    try:
        
        df_with_account = pd.read_excel("/Users/a0979/Desktop/tiktok/Data/7052019-09062024/Split_Data/with_account/Vooruit_with_account.xlsx") 
        
        usernames_list = []
        for url in df_with_account['Tiktok'].dropna():
            if '@' in str(url):
                username = str(url).split('@')[-1].split('/')[0].split('?')[0]
                usernames_list.append(username)
        usernames_list = list(set(usernames_list))
        
        
        test_usernames = usernames_list[:2]
        print(f"test usernames: {test_usernames}")
        
        scraping(
            query_list=test_usernames, 
            output_excel="Vooruit_test_official.xlsx",
            target_count=5,                            
            is_official_account=True,
            start_date='2019-05-27',                    
            end_date='2024-06-09'                      
        )
    except FileNotFoundError:
        print("can't find the file")

if __name__ == "__main__":
    main()

test usernames: ['hannelore.goeman', 'jorisvandenbroucke']

Search by account : 2 accounts
Looking for : hannelore.goeman
Looking for : jorisvandenbroucke
 5 videos, saved as：Vooruit_test_official.xlsx


In [50]:
import glob
import os

def main():

    input_folder = "/Users/a0979/Desktop/tiktok/Data/7052019-09062024/Split_Data/with_account/grouped_by_party"
    

    output_folder = "/Users/a0979/Desktop/tiktok/Data/7052019-09062024/Split_Data/with_account/Scraped_Results"
    
  
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Output folder：{output_folder}")

   
    search_pattern = os.path.join(input_folder, "*_with_account.xlsx")
    all_files = glob.glob(search_pattern)
    
    if not all_files:
        print(f"please check the path{input_folder}")
        return

    print(f"Here are {len(all_files)} files")
    for f in all_files:
        print(f" - {os.path.basename(f)}")
    print("-" * 50)


    for file_path in all_files:
        file_name = os.path.basename(file_path)
        party_name = file_name.replace("_with_account.xlsx", "")
        
        print(f"Working on：【{party_name}】")
        
        try:

            df_with_account = pd.read_excel(file_path)
            

            usernames_list = []
            if 'Tiktok' in df_with_account.columns:
                for url in df_with_account['Tiktok'].dropna():
                    if '@' in str(url):
                        username = str(url).split('@')[-1].split('/')[0].split('?')[0]
                        usernames_list.append(username)
                usernames_list = list(set(usernames_list))
            else:
                print(f"No 'Tiktok' column is in this file: {file_name} ")
                continue

            if not usernames_list:
                print(f"No videos for {party_name} ")
                continue

            print(f"{party_name} has {len(usernames_list)} usernames. ")
            
           
            output_excel_path = os.path.join(output_folder, f"{party_name}_official.xlsx")

            
            scraping(
                query_list=usernames_list, 
                output_excel=output_excel_path, 
                target_count=float('inf'), 
                is_official_account=True,
                start_date='2019-05-27', 
                end_date='2024-06-09'
            )
            
            
            time.sleep(5)

        except Exception as e:
            print(f" {file_name} Error: {e}")

if __name__ == "__main__":
    main()

Here are 7 files
 - Vooruit_with_account.xlsx
 - N-VA_with_account.xlsx
 - Open VLD_with_account.xlsx
 - Vlaams Belang_with_account.xlsx
 - Groen_with_account.xlsx
 - CDandV_with_account.xlsx
 - PVDA_with_account.xlsx
--------------------------------------------------
Working on：【Vooruit】
Vooruit has 6 usernames. 

Search by account : 6 accounts
Looking for : hannelore.goeman
Looking for : jorisvandenbroucke
Looking for : hannesanaf
Looking for : melissa.depraetere
Looking for : katiasegers
Looking for : caroline.gennez
 31 videos, saved as：/Users/a0979/Desktop/tiktok/Data/7052019-09062024/Split_Data/with_account/Scraped_Results/Vooruit_official.xlsx
Working on：【N-VA】
N-VA has 16 usernames. 

Search by account : 16 accounts
Looking for : yoleenvancamp
Looking for : ben_weyts
Looking for : franckentheo
Looking for : freyaperdaens
Looking for : sanderloones
Looking for : annickderidder.official
Looking for : bjanseeuw
Looking for : frieda.gijbels
Looking for : koen.metsu.tiktok
Looking f

In [52]:
import glob
import os

def main():

    input_folder = "/Users/a0979/Desktop/tiktok/Data/10062024 to present/split data/with_account/grouped_by_party"
    

    output_folder = "/Users/a0979/Desktop/tiktok/Data/10062024 to present/split data/with_account/Scraped_Results"
    
  
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Output folder：{output_folder}")

   
    search_pattern = os.path.join(input_folder, "*_with_account.xlsx")
    all_files = glob.glob(search_pattern)
    
    if not all_files:
        print(f"please check the path{input_folder}")
        return

    print(f"Here are {len(all_files)} files")
    for f in all_files:
        print(f" - {os.path.basename(f)}")
    print("-" * 50)


    for file_path in all_files:
        file_name = os.path.basename(file_path)
        party_name = file_name.replace("_with_account.xlsx", "")
        
        print(f"Working on：【{party_name}】")
        
        try:

            df_with_account = pd.read_excel(file_path)
            

            usernames_list = []
            if 'Tiktok' in df_with_account.columns:
                for url in df_with_account['Tiktok'].dropna():
                    if '@' in str(url):
                        username = str(url).split('@')[-1].split('/')[0].split('?')[0]
                        usernames_list.append(username)
                usernames_list = list(set(usernames_list))
            else:
                print(f"No 'Tiktok' column is in this file: {file_name} ")
                continue

            if not usernames_list:
                print(f"No videos for {party_name} ")
                continue

            print(f"{party_name} has {len(usernames_list)} usernames. ")
            
           
            output_excel_path = os.path.join(output_folder, f"{party_name}_official.xlsx")

            
            scraping(
                query_list=usernames_list, 
                output_excel=output_excel_path, 
                target_count=float('inf'), 
                is_official_account=True,
                start_date='2024-06-10', 
                end_date='2026-07-20'
            )
            
            
            time.sleep(5)

        except Exception as e:
            print(f" {file_name} Error: {e}")

if __name__ == "__main__":
    main()

Here are 8 files
 - Vooruit_with_account.xlsx
 - N-VA_with_account.xlsx
 - Open Vld_with_account.xlsx
 - Team Fouad Ahidar_with_account.xlsx
 - Vlaams Belang_with_account.xlsx
 - Groen_with_account.xlsx
 - cdandv_with_account.xlsx
 - PVDA_with_account.xlsx
--------------------------------------------------
Working on：【Vooruit】
Vooruit has 20 usernames. 

Search by account : 20 accounts
Looking for : frederiksioen
Looking for : hannesanaf
Looking for : katiasegers
Looking for : jorisvandenbroucke
Looking for : funda.oru
Looking for : oskar.seuntjens
Looking for : melissa.depraetere
Looking for : stephanie.vanden1
Looking for : axelweydts
Looking for : kellyvantendeloo
Looking for : hannelore.goeman
Looking for : kingconnah
Looking for : buraknalli
Looking for : jeroen.soete
Looking for : achrafelyakhloufi
Looking for : rob.beenders
Looking for : caroline.gennez
Looking for : alain.yzermans
Looking for : hiba__faraji
Looking for : nawal.mgh
 237 videos, saved as：/Users/a0979/Desktop/tikt

In [ ]:
    try:
        df_no_account = pd.read_excel("/Users/a0979/Desktop/tiktok/Data/7052019-09062024/Split_Data/no_account/PVDA_no_account.xlsx")
        keywords_list = df_no_account['search_keyword'].dropna().unique().tolist()
        
        scraping(
            query_list=keywords_list, 
            output_excel="Vlaams_Belang_keywords_search.excel", 
            target_count=1, 
            is_official_account=False,
            start_date='2019-05-27', 
            end_date='2024-06-07'
        )
    except FileNotFoundError:
        print("Can't find the file")